# 04 — Train the EGFR resistance model

This notebook trains and evaluates machine-learning models using:

- structural and physicochemical features,
- drug identity,
- experimental `auc_ratio_vs_wt` as the main continuous target, and
- the provided resistant/sensitive label as a secondary classification target.

## Validation strategy

Rows from the same mutation are kept together during cross-validation. This prevents the model from seeing one drug measurement for a mutation during training and another drug measurement for the same mutation during testing.

Because the dataset is small, the results should be treated as a pilot model rather than a clinically validated predictor.

## Step 1 — Upload the structural-feature CSV

Upload:

`egfr_structural_features.csv`


In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded))

## Step 2 — Install required packages

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib joblib

## Step 3 — Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv("egfr_structural_features.csv")

required = {"mutation", "drug", "auc_ratio_vs_wt", "resistant"}
missing = required.difference(data.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print("Rows:", len(data))
print("Distinct mutations:", data["mutation"].nunique())
print("Drugs:", sorted(data["drug"].unique()))
print("Resistant labels:")
print(data["resistant"].value_counts(dropna=False))

display(data.head())

## Step 4 — Select model features

Identifiers, target columns, ligand names, and bookkeeping columns are excluded from the numeric feature set. Drug identity is included through one-hot encoding.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

exclude = {
    "mutation", "pdb_id", "chain", "position", "wt_aa", "mut_aa",
    "auc_ratio_vs_wt", "resistant",
    "ligand_resname", "ligand_chain"
}

numeric_features = [
    c for c in data.columns
    if c not in exclude
    and pd.api.types.is_numeric_dtype(data[c])
]

categorical_features = ["drug"]

print("Numeric features:")
for feature in numeric_features:
    print(" -", feature)

print("\nCategorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "drug",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

## Step 5 — Build grouped cross-validation splits

A maximum of five folds is used, with grouping by mutation.

In [ ]:
from sklearn.model_selection import GroupKFold

X = data[numeric_features + categorical_features].copy()
y_reg = data["auc_ratio_vs_wt"].astype(float)
y_cls = data["resistant"].astype(int)
groups = data["mutation"].astype(str)

n_groups = groups.nunique()
n_splits = min(5, n_groups)

group_cv = GroupKFold(n_splits=n_splits)

print("Cross-validation folds:", n_splits)
print("Mutation groups:", n_groups)

## Step 6 — Train and evaluate the regression model

The regression model predicts the continuous AUC ratio relative to WT.

Metrics:

- **MAE:** average absolute prediction error
- **RMSE:** emphasizes larger errors
- **R²:** fraction of variance explained; it may be negative for weak small-sample models
- **Spearman correlation:** agreement in ranking resistant versus sensitive responses


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

regressor = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )),
])

reg_pred = cross_val_predict(
    regressor,
    X,
    y_reg,
    groups=groups,
    cv=group_cv,
    n_jobs=-1,
)

reg_metrics = {
    "MAE": float(mean_absolute_error(y_reg, reg_pred)),
    "RMSE": float(mean_squared_error(y_reg, reg_pred) ** 0.5),
    "R2": float(r2_score(y_reg, reg_pred)),
    "Spearman_r": float(spearmanr(y_reg, reg_pred).statistic),
}

print(reg_metrics)

regression_predictions = data[
    ["mutation", "drug", "auc_ratio_vs_wt", "resistant"]
].copy()
regression_predictions["predicted_auc_ratio_vs_wt"] = reg_pred
regression_predictions["absolute_error"] = np.abs(y_reg - reg_pred)

display(
    regression_predictions.sort_values(
        "absolute_error", ascending=False
    ).head(10)
)

## Step 7 — Plot observed versus predicted AUC ratio

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_reg, reg_pred)
low = min(y_reg.min(), reg_pred.min())
high = max(y_reg.max(), reg_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Observed AUC ratio vs WT")
plt.ylabel("Cross-validated predicted AUC ratio vs WT")
plt.title("EGFR TKI response: observed vs predicted")
plt.tight_layout()
plt.show()

## Step 8 — Train and evaluate the classification model

This model predicts the provided resistant/sensitive label.

Because the dataset may be imbalanced, balanced accuracy and ROC AUC are reported when both classes are available in each calculation.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

classifier = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

cls_pred = cross_val_predict(
    classifier,
    X,
    y_cls,
    groups=groups,
    cv=group_cv,
    method="predict",
    n_jobs=-1,
)

cls_prob = cross_val_predict(
    classifier,
    X,
    y_cls,
    groups=groups,
    cv=group_cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

cls_metrics = {
    "accuracy": float(accuracy_score(y_cls, cls_pred)),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_cls, cls_pred)
    ),
    "roc_auc": (
        float(roc_auc_score(y_cls, cls_prob))
        if y_cls.nunique() == 2 else None
    ),
}

print(cls_metrics)
print("\nConfusion matrix:")
print(confusion_matrix(y_cls, cls_pred))
print("\nClassification report:")
print(classification_report(y_cls, cls_pred, digits=3))

classification_predictions = data[
    ["mutation", "drug", "resistant", "auc_ratio_vs_wt"]
].copy()
classification_predictions["predicted_resistant"] = cls_pred
classification_predictions["resistance_probability"] = cls_prob

display(classification_predictions.head(10))

## Step 9 — Fit the final regression model on all available data

This fitted model is used for feature-importance analysis and future predictions.

In [ ]:
regressor.fit(X, y_reg)
classifier.fit(X, y_cls)

print("Final models fitted on all rows.")

## Step 10 — Rank predictive features with permutation importance

Permutation importance measures how much model performance worsens when one input column is randomly shuffled.

With a small dataset, importance rankings can be unstable. They should be interpreted as exploratory rather than definitive.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    regressor,
    X,
    y_reg,
    scoring="neg_mean_absolute_error",
    n_repeats=50,
    random_state=42,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance)

## Step 11 — Plot the top feature importances

In [ ]:
top = importance.head(15).sort_values("importance_mean")

plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["importance_mean"])
plt.xlabel("Permutation importance")
plt.ylabel("Feature")
plt.title("Structural features most predictive of AUC ratio")
plt.tight_layout()
plt.show()

## Step 12 — Save all outputs

The ZIP contains:

- regression metrics,
- classification metrics,
- cross-validated predictions,
- feature importance,
- the fitted regression model,
- the fitted classification model, and
- the list of model input columns.


In [ ]:
import json
import joblib
import shutil
from pathlib import Path

results_dir = Path("egfr_model_results")
if results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir()

with open(results_dir / "regression_metrics.json", "w") as f:
    json.dump(reg_metrics, f, indent=2)

with open(results_dir / "classification_metrics.json", "w") as f:
    json.dump(cls_metrics, f, indent=2)

regression_predictions.to_csv(
    results_dir / "regression_predictions.csv", index=False
)
classification_predictions.to_csv(
    results_dir / "classification_predictions.csv", index=False
)
importance.to_csv(
    results_dir / "feature_importance.csv", index=False
)

joblib.dump(regressor, results_dir / "auc_regression_model.joblib")
joblib.dump(classifier, results_dir / "resistance_classifier.joblib")

with open(results_dir / "model_columns.json", "w") as f:
    json.dump({
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
    }, f, indent=2)

archive = shutil.make_archive(
    "egfr_model_results",
    "zip",
    root_dir=results_dir,
)

print("Created:", archive)
files.download(archive)

## How to interpret the result

The main scientific answer comes from `feature_importance.csv`.

The strongest positive importance values identify the structural or physicochemical descriptors that most improve prediction of experimental AUC ratio.

However, the model currently covers only mutations that could be generated from the available kinase-domain crystal structures. It does not yet represent mutations outside the kinase domain.